 E-12 (main table): exact balance, inference, omitted direction, BP existence

Registration: `doc/2026-10-07_experiment_registration.md` (parent repository), E-12 and §1.4–§1.9; Amendment 4 for the Newton acceptance. One notebook for every stage; the runner passes the stage:

- `python3 notebooks/experiments/run_registered.py E-12 stage0 10_E12_omitted_direction_balance.ipynb`
- `... E-12 stage0.5 ...`, then `... E-12 stage1 ...` (each after the previous stage is committed).

DGP, arms and the replication function are in `grrexp/e12.py`. A solver that raises stops the notebook and the run is recorded as failed (§1.1).

In [ ]:
import collections
import json
import os
import time

import numpy as np
import pandas as pd

from grrexp import env, outputs

gr = env.use_submodule_genriesz()
from grrexp import bootstrap, e12, inference, metrics, parallel  # noqa: E402
from grrexp.seeds import CONFIRMATORY_ENTROPY, PILOT_ENTROPY, Seeds, fold_ids  # noqa: E402

STAGE = json.loads(os.environ[outputs.RUN_ENV])["stage"]
ARM_LABELS = [f"{a}|{est}" for a in e12.ALL_ARMS for est in ("RW_full", "ARW_cf", "TMLE_cf")] + [
    "UKL1|ARW_EF", "LogitAIPW", "DML-GBM"]
rec = outputs.RunRecorder(12, STAGE, cells=None if STAGE == "stage0" else e12.CELLS,
                          arms=None if STAGE == "stage0" else [a.replace("|", "-") for a in ARM_LABELS])
STAGE

# Stage 0: population quantities (registration E-12 Stage 0, §1.9)

In [ ]:
if STAGE == "stage0":
    rows, gate = e12.stage0()
    rec.write_stage0("e12_population", {"cells": rows, "reference_gate": gate})
    print(json.dumps(gate, indent=1))
    bad = [k for k, v in gate.items() if not v["ok"]]
    if bad:
        raise RuntimeError(f"Stage 0 reference gate failed: {bad}")  # registration: stop
    rec.finalize(R=None, n=None, warnings=0, failures=0)

# GBM tuning on pilot data (stream 5) for each (s, n)

In [ ]:
def tune_all(entropy):
    seeds = Seeds(12, entropy=entropy)
    params = {}
    for ci, (s, rs, n) in enumerate(e12.CELLS):
        if rs != "Include":
            continue  # tuning is per (s, n); the pilot sample of the (s, Include, n) cell is used
        D, Z, Y = e12.draw(seeds.pilot(ci), n, s)
        cv = fold_ids(n, 5, seeds.inner_cv(ci, 0))
        seed3 = int(seeds.baseline(ci, 0).integers(0, 2**31 - 1))
        from grrexp import baselines
        params[(s, n)] = baselines.tune_gbm(D, Z, Y, cv, seed3)
    return params


def tasks_for(entropy, reps, gbm):
    return [(ci, r, entropy, gbm[(s, n)]) for ci, (s, rs, n) in enumerate(e12.CELLS) for r in range(reps)]

# Stage 0.5: timing, status counts, and 1-versus-12-worker identity (no estimates kept)

In [ ]:
if STAGE == "stage0.5":
    parallel.configure_worker()
    gbm = tune_all(PILOT_ENTROPY)
    tasks = tasks_for(PILOT_ENTROPY, outputs.PILOT_REPS, gbm)
    timing = []
    serial = []
    for ci in range(len(e12.CELLS)):
        sub = [t for t in tasks if t[0] == ci]
        t0 = time.perf_counter()
        serial += parallel.run_tasks(e12.replicate, sub, 1)
        timing.append({"cell": e12.CELLS[ci], "workers": 1, "seconds": time.perf_counter() - t0})
    t0 = time.perf_counter()
    para = parallel.run_tasks(e12.replicate, tasks, outputs.PILOT_WORKERS)
    timing.append({"cell": e12.CELLS[0], "workers": outputs.PILOT_WORKERS, "seconds": time.perf_counter() - t0})
    digest = parallel.assert_bit_identical(serial, para, expected=len(tasks))
    counts = collections.Counter()
    for (ci, _, _, _), res in zip(tasks, serial):
        for label in ARM_LABELS:
            if label in res:
                counts[(json.dumps(e12.CELLS[ci]), label.replace("|", "-"), res[label]["status"])] += 1
    status_df = pd.DataFrame([{"cell": json.loads(c), "arm": a, "status": st, "count": k}
                              for (c, a, st), k in sorted(counts.items())])
    rec.write_pilot("timing.csv", pd.DataFrame(timing))
    rec.write_pilot("status_counts.csv", status_df)
    rec.record_parallel_identity(cells=e12.CELLS, reps=outputs.PILOT_REPS, n_jobs=outputs.PILOT_WORKERS,
                                 digest=digest)
    print(pd.DataFrame(timing).to_string(), "\n", status_df.groupby(["arm", "status"])["count"].sum())
    rec.finalize(R={"main": outputs.PILOT_REPS}, n={"main": list(e12.N_VALUES)}, warnings=0,
                 failures=int(sum(k for (c, a, st), k in counts.items() if st != "ok")))

# Stage 1: confirmatory Monte Carlo

In [ ]:
if STAGE == "stage1":
    parallel.configure_worker()
    gbm = tune_all(CONFIRMATORY_ENTROPY)
    tasks = tasks_for(CONFIRMATORY_ENTROPY, e12.R, gbm)
    t0 = time.perf_counter()
    results = parallel.run_tasks(e12.replicate, tasks, outputs.PILOT_WORKERS)
    print("seconds", time.perf_counter() - t0)
    raw_rows, n_warn = [], 0
    for (ci, rep, _, _), res in zip(tasks, results):
        s, rs, n = e12.CELLS[ci]
        n_warn += res["_n_warnings"]
        checks = {f"check_{k}": v for k, v in res["_checks"].items()}
        for label in ARM_LABELS:
            if label in res:
                r = res[label]
                raw_rows.append({"cell": ci, "s": s, "set": rs, "n": n, "rep": rep, "arm": label,
                                 **{k: r.get(k, np.nan) for k in ("estimate", "se", "max_abs_alpha", "imbalance",
                                                                  "ess", "weight_program_gap")},
                                 "status": r["status"], "weight_program_verdict": r.get("weight_program_verdict", ""),
                                 **checks})
    raw = pd.DataFrame(raw_rows)
    rec.write_raw(raw)
    raw.shape, n_warn

In [ ]:
if STAGE == "stage1":
    # H12-Bal (deterministic): any failure is an implementation error and stops the notebook
    chk = raw.filter(like="check_").drop(columns=["check_eb_dual_newton_max_diff"], errors="ignore")
    worst = chk.abs().max().to_dict()
    eb = raw["check_eb_dual_newton_max_diff"].dropna() if "check_eb_dual_newton_max_diff" in raw else pd.Series([])
    rw_ok = raw[raw["arm"].str.endswith("RW_full") & (raw["status"] == "ok")]
    tol = 1e-8 * max(1.0, 1.0)
    bal = {"identities_max": worst, "eb_max": float(eb.max()) if len(eb) else None,
           "imbalance_max": float(rw_ok["imbalance"].max())}
    assert all(v <= 1e-10 for v in worst.values()), bal
    assert len(eb) == 0 or eb.max() <= 1e-6, bal
    pop = json.loads((outputs.stage_dir(12, "stage0") / "stage0_e12_population.json").read_text())["cells"]
    POP = {(r["arm"], r["set"], r["s"]): r for r in pop}
    bal

In [ ]:
if STAGE == "stage1":
    rows = []
    for (ci, label), g in raw.groupby(["cell", "arm"], sort=False):
        s, rs, n = e12.CELLS[ci]
        ok = (g["status"] == "ok").to_numpy()
        est, se = g["estimate"].to_numpy(), g["se"].to_numpy()
        row = {"cell": ci, "s": s, "set": rs, "n": n, "arm": label, **metrics.failure_rate(ok)}
        if ok.sum() >= 2:
            row.update(metrics.summarise(est, se, ok, e12.THETA0, n))
            if g["max_abs_alpha"].notna().any():
                row.update(metrics.max_weight_summary(g["max_abs_alpha"].fillna(0).to_numpy(), ok))
        arm = label.split("|")[0]
        p = POP.get(("UKL1" if label == "UKL1|ARW_EF" else arm, rs, s))
        if p is not None:
            row.update({"certified": p["certified"], "b_pred": p["b"],
                        "root_n_sd_pred": float(np.sqrt(p["sigma2_true"])), "c_pred": p["c_n"][str(n)],
                        "sigma_true": float(np.sqrt(p["sigma2_true"]))})
        row["status_counts"] = json.dumps(collections.Counter(g["status"]))
        rows.append(row)
    summ_df = pd.DataFrame(rows)

    def tests_for(sub):
        tests = {}
        for _, r in sub.iterrows():
            key = f"{r['arm']}|s={r['s']}|{r['set']}|n={r['n']}"
            g = raw[(raw["cell"] == r["cell"]) & (raw["arm"] == r["arm"])]
            ok = (g["status"] == "ok").to_numpy()
            x = g.loc[ok, "estimate"].to_numpy()
            tests[key + "|bias"] = inference.bias_pvalue(x, e12.THETA0, r["b_pred"], r["sigma_true"], r["n"])
            tests[key + "|sd_ratio"] = inference.sd_ratio_pvalue(x, r["sigma_true"], r["n"])
            tests[key + "|coverage"] = inference.coverage_pvalue(int(r["covered"]), int(r["R"]), r["c_pred"])
            tests[key + "|failure"] = inference.failure_rate_pvalue(int(r["R"] - r["R_s"]), int(r["R"]), 0.01)
        return tests

    big = summ_df["n"].isin([2000, 4000]) & (summ_df["certified"] == True)  # noqa: E712
    grr_rw_arw = summ_df["arm"].isin([f"{a}|{e}" for a in e12.GRR_ARMS for e in ("RW_full", "ARW_cf")])
    fam = {}
    fam["H12-Inf"] = inference.judge_family("H12-Inf", tests_for(summ_df[big & grr_rw_arw & (summ_df["set"] == "Include")]))
    fam["H12-Bias"] = inference.judge_family("H12-Bias", tests_for(summ_df[big & grr_rw_arw & (summ_df["set"] == "Omit")]))
    fam["H12-EF"] = inference.judge_family("H12-EF", tests_for(summ_df[big & (summ_df["arm"] == "UKL1|ARW_EF")]))
    bp = summ_df[(summ_df["n"] == 4000) & (summ_df["certified"] == True)  # noqa: E712
                 & summ_df["arm"].isin(["BP_C1|RW_full", "BP_C1|ARW_cf", "BP_C0|RW_full", "BP_C0|ARW_cf"])]
    fam["H12-BP"] = inference.judge_family("H12-BP", {
        f"{r['arm']}|s={r['s']}|{r['set']}": inference.failure_rate_pvalue(int(r["R"] - r["R_s"]), int(r["R"]), 0.01)
        for _, r in bp.iterrows()}) if len(bp) else None
    {k: (v.sentence() if v else "no certified BP cell") for k, v in fam.items()}

In [ ]:
if STAGE == "stage1":
    # H12-Ord: s = 1.25, Include, n = 4000; replications where SQ, BP(C=1), UKL, BKL all succeed (RW_full)
    ci = e12.CELLS.index([1.25, "Include", 4000])
    order = ["SQ", "BP_C1", "UKL1", "BKL1"]
    wide = raw[(raw["cell"] == ci) & raw["arm"].isin([f"{a}|RW_full" for a in order])].pivot(
        index="rep", columns="arm", values="max_abs_alpha")
    stat = raw[(raw["cell"] == ci) & raw["arm"].isin([f"{a}|RW_full" for a in order])].pivot(
        index="rep", columns="arm", values="status")
    all_ok = (stat == "ok").all(axis=1)
    share = float(all_ok.mean())
    ord_rows = {"share_all_succeeded": share, "pairs": []}
    if share >= 0.5:
        mw = wide[all_ok][[f"{a}|RW_full" for a in order]].to_numpy()
        fb = bootstrap.FamilyBootstrap(Seeds(12), 3)
        passes = []
        for j in range(3):
            diff = lambda idx, j=j: np.median(mw[idx, j + 1], axis=1) - np.median(mw[idx, j], axis=1)  # noqa: E731
            boot = fb.replicate(diff, mw.shape[0])
            lo, hi = bootstrap.percentile_interval(boot, 3)
            point = float(np.median(mw[:, j + 1]) - np.median(mw[:, j]))
            ord_rows["pairs"].append({"pair": f"{order[j]}<{order[j + 1]}", "median_diff": point, "ci": [lo, hi]})
            passes.append(lo > 0)
        ord_rows["verdict"] = "pass" if all(passes) else "fail"
    else:
        ord_rows["verdict"] = "undeterminable"
    summ_df["family_verdicts"] = json.dumps({k: (v.sentence() if v else None) for k, v in fam.items()})
    summ_df["ord"] = json.dumps(ord_rows)
    summ_df["h12_bal"] = json.dumps(bal)
    rec.write_summary(summ_df)
    ord_rows

# Tables and macros (from summary.csv only, §4)

In [ ]:
if STAGE == "stage1":
    S = outputs.read_summary(rec)
    LABEL = {"SQ": "SQ", "UKL1": "UKL ($C=1$)", "BKL1": "BKL ($C=1$)", "BP_C1": "BP ($C=1$)",
             "BP_C0": "BP ($C=0$)", "EB": "EB (= UKL, $C=0$)", "LogitAIPW": "LogitAIPW", "DML-GBM": "DML-GBM"}
    END = " \\\\"  # LaTeX row end

    def fmt(x, d=3):
        return "--" if x != x else f"{x:.{d}f}"

    def tex(x):
        return str(x).replace("_", "\\_").replace("|", " ").replace("%", "\\%").replace("<", "$<$")

    def table(df, cols, heads, spec):
        lines = ["\\begin{tabular}{" + spec + "}", "\\hline", " & ".join(heads) + END, "\\hline"]
        for _, r in df.iterrows():
            lines.append(" & ".join(fmt(r[c]) if isinstance(r[c], float) else tex(r[c]) for c in cols) + END)
        return "\n".join(lines + ["\\hline", "\\end{tabular}"]) + "\n"

    def main_rows(set_, s, estimator):
        out = []
        for arm in ("SQ", "UKL1", "BKL1", "BP_C1", "BP_C0", "EB", "LogitAIPW", "DML-GBM"):
            label = arm if arm in ("LogitAIPW", "DML-GBM") else f"{arm}|{estimator}"
            cells_ = []
            for n in (2000, 4000):
                r = S[(S["arm"] == label) & (S["set"] == set_) & (S["s"] == s) & (S["n"] == n)]
                if r.empty:
                    cells_ += ["--"] * 5
                    continue
                r = r.iloc[0]
                cells_ += [f"{fmt(r.get('bias', np.nan))} ({fmt(r.get('b_pred', np.nan))})",
                           f"{fmt(r.get('root_n_sd', np.nan), 2)} ({fmt(r.get('root_n_sd_pred', np.nan), 2)})",
                           f"{fmt(r.get('coverage', np.nan))} ({fmt(r.get('c_pred', np.nan))})",
                           f"{100 * r['failure_rate']:.1f}", fmt(r.get('max_weight_median', np.nan), 1)]
            out.append(" & ".join([LABEL[arm]] + cells_) + END)
        return out

    heads = ["Method"] + ["Bias ($b$)", "$\\sqrt{n}$SD (pred.)", "Coverage ($c(n)$)", "Fail \\%",
                          "med. $\\max|\\widehat\\alpha|$"] * 2
    lines = ["\\begin{tabular}{l" + "rrrrr" * 2 + "}", "\\hline",
             " & \\multicolumn{5}{c}{$n=2000$} & \\multicolumn{5}{c}{$n=4000$}" + END, " & ".join(heads) + END,
             "\\hline"]
    for s in e12.S_VALUES:
        for set_ in e12.SETS:
            lines.append("\\multicolumn{11}{l}{$s=" + str(s) + "$, " + set_ + "}" + END)
            lines += main_rows(set_, s, "ARW_cf")
    rec.write_table("tab_E12_main", "\n".join(lines + ["\\hline", "\\end{tabular}"]) + "\n")

    full_cols = ["arm", "s", "set", "n", "bias", "b_pred", "root_n_sd", "root_n_sd_pred", "se_ratio", "coverage",
                 "c_pred", "ci_length_mean", "failure_rate", "max_weight_median"]
    full = S[[c for c in full_cols if c in S.columns]]
    rec.write_table("tab_E12_full", table(full, list(full.columns), [tex(c) for c in full.columns],
                                          "llll" + "r" * (len(full.columns) - 4)))
    bp_tab = S[S["arm"].str.startswith("BP")].copy()
    bp_tab["statuses"] = [", ".join(f"{k}: {v}" for k, v in json.loads(c).items()) for c in bp_tab["status_counts"]]
    rec.write_table("tab_E12_bp", table(bp_tab, ["arm", "s", "set", "n", "failure_rate", "statuses"],
                                        ["Arm", "$s$", "Set", "$n$", "Failure rate", "Statuses"], "llllrl"))
    ordr = json.loads(S["ord"].iloc[0])
    ord_df = pd.DataFrame([{"pair": p["pair"], "diff": p["median_diff"], "lo": p["ci"][0], "hi": p["ci"][1]}
                           for p in ordr.get("pairs", [])], columns=["pair", "diff", "lo", "hi"])
    rec.write_table("tab_E12_ord", table(ord_df, ["pair", "diff", "lo", "hi"],
                                         ["Pair", "Median difference", "Lower 95\\%", "Upper 95\\%"], "lrrr"))
    fv = json.loads(S["family_verdicts"].iloc[0])

    def verdict_word(k, v):
        if v is None:
            return "none"
        return "negative" if v.startswith(k + ": negative") else "no departure detected"

    rec.write_macros({
        "EXIIR": str(e12.R),
        "EXIIOrdVerdict": ordr["verdict"],
        "EXIIOrdShare": f"{ordr['share_all_succeeded']:.3f}",
        **{"EXII" + k.replace("H12-", "").replace("-", "") + "Verdict": verdict_word(k, v) for k, v in fv.items()},
    })
    n_fail = int((S["failure_rate"] * S["R"]).sum())
    rec.finalize(R={"main": e12.R}, n={"main": list(e12.N_VALUES)}, warnings=int(n_warn), failures=n_fail)